In [1]:

from pathlib import Path
import subprocess, sys, os

REPO = Path('/content/Bottlevision')
REMOTE = 'https://github.com/Rollerboy22/Bottlevision.git'

if REPO.exists():
    subprocess.run(['rm', '-rf', str(REPO)], check=True)

subprocess.run(['git', 'clone', '--branch', 'main', '--depth', '1', REMOTE, str(REPO)], check=True)
%cd /content/Bottlevision
print("Commit:", subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print("Colab Python:", sys.version.split()[0])

/content/Bottlevision
Commit: cbaf608
Colab Python: 3.13.15


In [2]:
import subprocess, sys
from pathlib import Path

# Ставим uv
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True)
UV = subprocess.check_output(['bash', '-lc', 'command -v uv || echo /root/.local/bin/uv'], text=True).strip()

VENV = REPO / '.venv312'
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
subprocess.run([UV, 'venv', '--python', '3.12', str(VENV), '--clear'], check=True)

PYTHON = VENV / 'bin' / 'python'
print("ML Python:", subprocess.check_output([str(PYTHON), '-c', 'import sys; print(sys.version)'], text=True).strip())

ML Python: 3.12.14 (main, Sep  1 2026, 14:16:52) [Clang 22.1.3 ]


In [3]:
import subprocess
from pathlib import Path

UV = subprocess.check_output(['bash', '-lc', 'command -v uv || echo /root/.local/bin/uv'], text=True).strip()
PYTHON = str(REPO / '.venv312' / 'bin' / 'python')

# PyTorch + CUDA
subprocess.run([
    UV, 'pip', 'install', '--python', PYTHON,
    'torch==2.10.0', 'torchvision==0.25.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'
], check=True)

# Остальные зависимости + SAM 3
subprocess.run([
    UV, 'pip', 'install', '--python', PYTHON,
    'numpy==1.26.4',
    'pydantic>=2.7,<3', 'PyYAML>=6.0,<7', 'Pillow>=10,<12',
    'timm>=1.0.17', 'tqdm', 'ftfy==6.1.1', 'regex',
    'iopath>=0.1.10', 'typing_extensions', 'huggingface_hub>=0.30',
    'einops>=0.8',
    'git+https://github.com/facebookresearch/sam3.git'
], check=True)

# Bottle Vision (просто добавляем путь, без editable install)
print("Installation finished.")

Installation finished.


In [ ]:

import subprocess
from pathlib import Path

UV = subprocess.check_output(['bash', '-lc', 'command -v uv || echo /root/.local/bin/uv'], text=True).strip()
PYTHON = str(REPO / '.venv312' / 'bin' / 'python')

subprocess.run([
    UV, 'pip', 'install', '--python', PYTHON,
    'pycocotools'
], check=True)

print("pycocotools установлен")

pycocotools установлен


In [ ]:
import subprocess
from pathlib import Path

UV = subprocess.check_output(['bash', '-lc', 'command -v uv || echo /root/.local/bin/uv'], text=True).strip()
PYTHON = str(REPO / '.venv312' / 'bin' / 'python')

subprocess.run([
    UV, 'pip', 'install', '--python', PYTHON,
    'pycocotools',
    'psutil',
    'opencv-python-headless',
    'matplotlib',
    'scikit-image'
], check=True)

print("Дополнительные зависимости установлены")

Дополнительные зависимости установлены


In [ ]:
import subprocess
from pathlib import Path

PYTHON = str(REPO / '.venv312' / 'bin' / 'python')
SRC = str(REPO / 'src')

check = f'''
import sys
sys.path.insert(0, "{SRC}")

import numpy, torch, torchvision, sam3, bottle_vision
print("Python     :", sys.version.split()[0])
print("NumPy      :", numpy.__version__)
print("PyTorch    :", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
print("SAM3       : OK")
print("bottle_vision: OK")
print("\\nВсе готово!")
'''

result = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Exit code:", result.returncode)

Python     : 3.12.14
NumPy      : 2.5.3
PyTorch    : 2.10.0+cu128
TorchVision: 0.25.0+cu128
CUDA       : True
GPU        : Tesla T4
SAM3       : OK
bottle_vision: OK

Все готово!

STDERR: /content/Bottlevision/.venv312/lib/python3.12/site-packages/sam3/model/model_misc.py:71: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
/content/Bottlevision/.venv312/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

Exit code: 0


In [ ]:
from pathlib import Path
import subprocess
import os

REPO = Path('/content/Bottlevision')
PYTHON = str(REPO / '.venv312' / 'bin' / 'python')

# Берём токен из Colab Secret (рекомендуется) или переменной окружения.
# В Colab: 🔑 Secrets -> добавить HF_TOKEN.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError(
        'Не найден HF_TOKEN. Добавь HF_TOKEN в Colab Secrets (значок 🔑 слева) '
        'и включи доступ для этого notebook.'
    )

# Авторизуемся внутри venv.
subprocess.run([
    PYTHON, '-c',
    'from huggingface_hub import login; import os; login(token=os.environ["HF_TOKEN"]); print("Logged in successfully")'
], env={**os.environ, 'HF_TOKEN': HF_TOKEN}, check=True)

print('Авторизация Hugging Face прошла')


Авторизация прошла


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

REPO = Path('/content/Bottlevision')

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Картинка не загружена")

filename, data = next(iter(uploaded.items()))
dst_path = REPO / 'colab_input.jpg'
dst_path.write_bytes(data)

print("Получен файл:", filename)
print("Сохранено в:", dst_path)
print("Размер:", dst_path.stat().st_size, "байт")

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/Bottlevision')
PYTHON = str(REPO / '.venv312' / 'bin' / 'python')
SRC = str(REPO / 'src')
dst = REPO / 'colab_input.jpg'

run_script = REPO / 'colab' / '_run_once.py'
run_script.write_text(f'''
import sys
sys.path.insert(0, "{SRC}")

import json
from pathlib import Path
import numpy as np
from PIL import Image
import torch

torch.set_default_dtype(torch.float32)

# T4 fix: force SAM3 image inference to Float32.
import sam3.model_builder as mb
_original_build = mb.build_sam3_image_model

def _build_float32(*args, **kwargs):
    model = _original_build(*args, **kwargs)
    return model.float().eval()

mb.build_sam3_image_model = _build_float32

from sam3.model.sam3_image_processor import Sam3Processor
import torchvision.transforms.v2 as v2
import PIL

@torch.inference_mode()
def _set_image_float32(self, image, state=None):
    if state is None:
        state = {{}}

    if isinstance(image, PIL.Image.Image):
        width, height = image.size
    elif isinstance(image, (torch.Tensor, np.ndarray)):
        height, width = image.shape[-2:]
    else:
        raise ValueError("Image must be a PIL image or a tensor")

    image = v2.functional.to_image(image).to(self.device)
    image = self.transform(image).unsqueeze(0).float()
    self.model = self.model.float()

    state["original_height"] = height
    state["original_width"] = width
    state["backbone_out"] = self.model.backbone.forward_image(image)

    inst_interactivity_en = self.model.inst_interactive_predictor is not None
    if inst_interactivity_en and "sam2_backbone_out" in state["backbone_out"]:
        sam2_backbone_out = state["backbone_out"]["sam2_backbone_out"]
        sam2_backbone_out["backbone_fpn"][0] = self.model.inst_interactive_predictor.model.sam_mask_decoder.conv_s0(
            sam2_backbone_out["backbone_fpn"][0]
        )
        sam2_backbone_out["backbone_fpn"][1] = self.model.inst_interactive_predictor.model.sam_mask_decoder.conv_s1(
            sam2_backbone_out["backbone_fpn"][1]
        )

    return state

Sam3Processor.set_image = _set_image_float32

from bottle_vision.config import load_config
from bottle_vision.pipeline import run_pipeline
from bottle_vision.segmentation import make_review_views

image_path = Path("{dst}")
output_dir = Path("/content/Bottlevision/colab_output")
output_dir.mkdir(exist_ok=True)

config = load_config(Path("/content/Bottlevision/configs/default.yaml"))
image = np.asarray(Image.open(image_path).convert("RGB"), dtype=np.uint8)
print("Shape:", image.shape)

print("Запускаю SAM3 один раз (Float32 / T4-safe)...")
with torch.autocast(device_type="cuda", enabled=False):
    result = run_pipeline(image, config)

seg = result.segmentation
print("Модель:", seg.model_name)
print("Инстансов:", len(seg.instances))
if seg.error:
    print("Ошибка:", seg.error)

summary = {{
    "model": seg.model_name,
    "instance_count": len(seg.instances),
    "error": seg.error,
    "instances": [
        {{
            "id": inst.instance_id,
            "confidence": float(inst.confidence) if inst.confidence is not None else None,
            "accepted": bool(inst.metadata.get("accepted_by_gate", False)),
        }}
        for inst in seg.instances
    ]
}}
(output_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))

views = make_review_views(image, seg, alpha=0.45)
for name, arr in views.items():
    safe = name.lower().replace(" ", "_")
    Image.fromarray(np.asarray(arr, dtype=np.uint8)).save(output_dir / f"{{safe}}.png")
    print("Сохранено:", safe + ".png")

print("Готово")
''', encoding='utf-8')

print("Запуск...")
result = subprocess.run([PYTHON, str(run_script)], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    err = result.stderr
    print("STDERR (последние 2000):")
    print(err[-2000:] if len(err) > 2000 else err)
print("Exit code:", result.returncode)
# Предпросмотр прямо в Colab
from IPython.display import display, Image as DisplayImage

preview_files = [
    ("Входное изображение", REPO / "colab_input.jpg"),
    ("Результат: overlay", REPO / "colab_output" / "overlay.png"),
    ("Маска", REPO / "colab_output" / "mask.png"),
]

print("\nПредпросмотр:")
for title, path in preview_files:
    if path.exists():
        print(f"\n{title}")
        display(DisplayImage(filename=str(path), width=900))
    else:
        print(f"Не найден файл: {path}")
